# NOTEBOOK: 02_silver_standardize


- Cleans the messy weight_raw column → converts everything to kg as a float
- Cleans the messy unit_price_raw column → converts everything to USD as a float
- Deduplicates vendor names (Sony Corp. + sony corporation → Sony Corporation)

# 0. Imports and Configs

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
from delta.tables import DeltaTable
from datetime import datetime


import re
from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType


In [0]:
# Source Bronze tables
BRONZE_PRODUCTS = "main.bronze_techmart.raw_products"
BRONZE_VENDORS  = "main.bronze_techmart.raw_vendors"

# Target Silver tables
SILVER_PRODUCTS = "main.silver_techmart.products"
SILVER_VENDORS  = "main.silver_techmart.vendors"

PROCESSED_AT = datetime.now().isoformat()

print("✅ Config loaded")

In [0]:
# Read Bronze tables
df_products = spark.table(BRONZE_PRODUCTS)
df_vendors  = spark.table(BRONZE_VENDORS)

In [0]:
print(f"Bronze vendors : {df_vendors.count()} rows")
df_vendors.head()

In [0]:
print(f"Bronze products: {df_products.count()} rows")
df_products.display()

# 1. Clean Product

## 1.1 Normalize weight to kg

In [0]:
# The weight column has many formats: "12.5 kg", "250 grams", "370gr", "5.8 Kilograms", "32.9 grs", "141 G.", "6.8 g" etc.
# Strategy: extract the numeric value, detect the unit, convert to kg.

def parse_weight_to_kg(raw):
    """
    Parses any weight string and returns a float in kg.
    Handles: kg, g, grams, gr, grs, kilograms, G, Kg, KG etc.
    Returns None if the value cannot be parsed.
    """
    if raw is None or str(raw).strip().lower() in ("none", "nan", ""):
        return None
    
    raw = str(raw).strip().lower()
    
    # Extract the numeric part (handles: 12.5, 1,8, 12 5 with spaces)
    # Remove currency symbols and text, keep digits, dots, commas
    numeric_str = re.sub(r"[^\d.,]", " ", raw).strip()
    # Handle comma as decimal separator and remove spaces between digits
    numeric_str = numeric_str.replace(",", ".").replace(" ", "")
    
    try:
        value = float(numeric_str)
    except:
        return None
    
    # Detect unit and convert to kg
    if any(unit in raw for unit in ["kilogram", "kilograms"]):
        return round(value, 4)
    elif raw.endswith("kg") or " kg" in raw:
        return round(value, 4)
    elif any(unit in raw for unit in ["gram", "grams", "gr", "grs", "g.", " g"]):
        return round(value / 1000, 4)
    elif raw[-1] == "g" or raw.endswith("kg"):
        # ends with just "g" like "900g" or "370gr"
        if "kg" in raw:
            return round(value, 4)
        else:
            return round(value / 1000, 4)
    else:
        # Default: if unit unclear but value seems reasonable for kg, keep as is
        return round(value, 4)
    
# Register as a Spark UDF so we can apply it to the whole column at once
parse_weight_udf = udf(parse_weight_to_kg, DoubleType())


In [0]:
# Apply the UDF
df_products = df_products.withColumn(
    "weight_kg",
    parse_weight_udf(F.col("weight_raw"))
)

print("✅ Weight normalized")
display(df_products.select("product_id", "product_description", "weight_raw", "weight_kg"))

## 1.2 Normalize price to USD float

In [0]:
# Normalize price to USD float
# Formats found: "$499.99", "749", "999.00 USD", "59 99", "$4 50.00", "349,00 usd"
# Strategy: strip all non-numeric characters carefully, handle edge cases

def parse_price_to_usd(raw):
    """
    Parses any price string and returns a float in USD.
    Handles: $, USD, spaces as thousands/decimal separators, commas.
    Returns None if the value cannot be parsed.
    """
    if raw is None or str(raw).strip().lower() in ("none", "nan", ""):
        return None
    
    raw = str(raw).strip().lower()
    
    # Remove known text: $, usd, dollars and surrounding spaces
    raw = raw.replace("$", "").replace("usd", "").replace("dollars", "").strip()
    
    # Handle comma as decimal separator (e.g. "349,00")
    # Rule: if comma exists and only 2 digits follow it → decimal separator
    if "," in raw and "." not in raw:
        parts = raw.split(",")
        if len(parts) == 2 and len(parts[1].strip()) <= 2:
            raw = raw.replace(",", ".")
        else:
            raw = raw.replace(",", "")
    
    # Remove all remaining spaces (e.g. "59 99" → "5999"... wait, that's wrong)
    # "59 99" should be "59.99" — space is acting as decimal separator here
    # Detect: if space exists and exactly 2 digits follow the space → decimal
    space_decimal = re.match(r"^(\d+)\s(\d{2})$", raw.strip())
    if space_decimal:
        raw = f"{space_decimal.group(1)}.{space_decimal.group(2)}"
    else:
        raw = raw.replace(" ", "")
    
    try:
        return round(float(raw), 2)
    except:
        return None

parse_price_udf = udf(parse_price_to_usd, DoubleType())


In [0]:

df_products = df_products.withColumn(
    "unit_price_usd",
    parse_price_udf(F.col("unit_price_raw"))
)

print("✅ Price normalized")
display(df_products.select("product_id", "product_description", "unit_price_raw", "unit_price_usd"))

# 2. Clean Vendors

In [0]:
# Deduplicate vendor names
# Problem: same vendor appears with different spellings
# "Sony corporation", "Sony Corp." → both mean Sony
# Strategy: normalize to title case and map known duplicates to a canonical name

def normalize_vendor(raw):
    """
    Normalizes vendor names to a canonical form.
    Handles case differences, punctuation, and known abbreviations.
    """
    if raw is None or str(raw).strip().lower() in ("none", "nan", ""):
        return None
    
    # Normalize to title case first
    normalized = str(raw).strip().title()
    
    # Manual canonical mapping for known duplicates in this dataset
    canonical_map = {
        # Sony variants
        "Sony Corporation"       : "Sony Corporation",
        "Sony Corp."             : "Sony Corporation",
        "Sony Corp"              : "Sony Corporation",
        # Samsung variants
        "Samsung Electronics Co.": "Samsung Electronics",
        "Samsung Electronics"    : "Samsung Electronics",
        "Samsung Electronics Co" : "Samsung Electronics",
        # Apple variants
        "Apple Inc"              : "Apple Inc.",
        "Apple Inc."             : "Apple Inc.",
        "Apple Inc.."            : "Apple Inc.",
        # Lenovo variants
        "Lenovo Group Ltd"       : "Lenovo Group",
        "Lenovo Group Ltd."      : "Lenovo Group",
        # Asus variants
        "Asus Tek Computer Inc." : "Asus",
        "Asus Tek Computer Inc"  : "Asus",
        # Nintendo variants
        "Nintendo Co Ltd"        : "Nintendo",
        "Nintendo Co. Ltd"       : "Nintendo",
        # Hyperx / Kingston
        "Hyperx  (Kingston)"     : "HyperX (Kingston)",
        "Hyperx (Kingston)"      : "HyperX (Kingston)",
    }
    
    return canonical_map.get(normalized, normalized)

normalize_vendor_udf = udf(normalize_vendor)

df_vendors = df_vendors.withColumn(
    "vendor_name",
    normalize_vendor_udf(F.col("vendor_name_raw"))
)

print("✅ Vendor names normalized")
display(df_vendors.select("product_id", "vendor_name_raw", "vendor_name"))